# Monte Carlo pricing and the implied-volatility smile

This notebook uses the `volsurface` package to connect three ideas: **Monte Carlo prices**, the
**Black-Scholes implied-volatility solver** and the **volatility smile** that stochastic volatility
produces. Install the plotting dependency with `uv sync --group notebooks`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from volsurface.pricing.black_scholes import bs_price
from volsurface.pricing.implied_vol import implied_vol_bs
from volsurface.pricing.montecarlo import (
    GbmModel,
    HestonModel,
    MonteCarloEngine,
    Payoff,
    PayoffKind,
    TwoHestonModel,
)

S0, R, Q, T = 100.0, 0.03, 0.01, 1.0

## 1. GBM Monte Carlo against Black-Scholes

Under geometric Brownian motion the closed form is available, so the estimator can be checked
directly. The deviation should be a few standard errors at most.

In [ ]:
sigma = 0.25
engine = MonteCarloEngine(GbmModel(S0, R, Q, sigma))
print(f"{'strike':>7} {'MC price':>10} {'std err':>9} {'BS price':>10} {'dev (s.e.)':>11}")
for strike in (80, 90, 100, 110, 120):
    mc = engine.price(Payoff(PayoffKind.CALL, strike), T, 400_000, seed=1)
    exact = float(bs_price(S0, strike, T, R, Q, sigma, True))
    print(
        f"{strike:7d} {mc.price:10.4f} {mc.std_error:9.4f} {exact:10.4f} {(mc.price - exact) / mc.std_error:11.2f}"
    )

## 2. Convergence

The running estimate and its two-standard-error band narrow as $1/\sqrt n$.

In [ ]:
result = engine.price(Payoff(PayoffKind.CALL, 100.0), T, 1_000_000, seed=2, checkpoints=80)
exact = float(bs_price(S0, 100.0, T, R, Q, sigma, True))
shown = result.counts >= 2_000
n, price, se = result.counts[shown], result.running_price[shown], result.running_std_error[shown]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(n, price, label="Monte Carlo")
ax.fill_between(n, price - 2 * se, price + 2 * se, alpha=0.25, label="+/- 2 s.e.")
ax.axhline(exact, color="tab:orange", linestyle="--", label="Black-Scholes")
ax.set_xscale("log")
ax.set_xlabel("paths")
ax.set_ylabel("call price")
ax.legend()
fig.tight_layout()

## 3. Heston prices through the implied-volatility solver

Price calls across strikes with a stochastic-volatility model, then invert each price with the
Black-Scholes solver. A flat line would mean Black-Scholes is right; the slope is the smile that
negative spot-vol correlation creates.

In [ ]:
heston = MonteCarloEngine(
    HestonModel(S0, 0.04, R, Q, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7), steps=100
)
strikes = np.array([80.0, 85, 90, 95, 100, 105, 110, 115, 120])
prices = np.array(
    [heston.price(Payoff(PayoffKind.CALL, k), T, 200_000, seed=3).price for k in strikes]
)
smile = 100 * implied_vol_bs(prices, S0, strikes, T, R, Q, True)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(np.log(strikes / (S0 * np.exp((R - Q) * T))), smile, "o-")
ax.set_xlabel("log-moneyness")
ax.set_ylabel("implied vol %")
ax.set_title("Smile implied by Heston Monte Carlo prices")
fig.tight_layout()
smile

## 4. A rainbow (best-of) call

Two Heston assets with correlated price shocks. Higher correlation makes the best-of option cheaper,
because the assets move together and there is less to choose between.

In [ ]:
for rho_assets in (0.0, 0.5, 0.9):
    model = TwoHestonModel(S0, 0.04, R, Q, 1.5, 0.04, 0.3, -0.7, rho_assets)
    out = MonteCarloEngine(model, steps=60).price(
        Payoff(PayoffKind.BEST_OF_CALL, 100.0), T, 100_000, seed=4
    )
    print(f"asset correlation {rho_assets:.1f}: price {out.price:.4f} +/- {out.std_error:.4f}")